# Who Goes There: train one config on Kaggle

Trains one config of `config.yaml` (the small CNN, ResNet-18 arm A or arm B) on the 512 px CCT20
cache with `trailcam.train.train`, then prints the ledger row and the checkpoint's SHA-256 (item
6.02; the recipe is `docs/plan.md` sections 5 and 6). The code, `config.yaml` and the ledger all
come from one clone of the public repo at the commit `COMMIT`, so the config hash in the row is
the hash of that commit's config. This notebook never scores cis_test or trans_test: that is the
final scoring (items 9.01 and 9.02), with the user's go-ahead.

## How to run on Kaggle

1. Import this notebook into Kaggle (File > Import Notebook: the file, or its GitHub link).
2. Add Input: attach the private 512 px cache dataset of item 6.03 (`annotations/`, `images/` and
   `day_night.json`).
3. Set the parameters in the first code cell: `CONFIG_NAME`, `RUN_SEED` (`None` keeps the
   config's seed), `COMMIT` (the full sha of the `main` commit to run) and `DATA_ROOT` (the
   folder of the attached dataset that holds the three; the check cell lists candidates if it
   is wrong).
4. Session options: Accelerator GPU, Internet on (the clone, pip and the ImageNet weights).
5. Save Version > Save & Run All (Commit). The checkpoint (`/kaggle/working/models/`) and the
   clone with its ledger are kept as the version's output.
6. Paste the printed ledger row into `docs/ledger.md` (items 6.04 and 6.05) and note the commit
   sha in the progress-log entry. If the training cell stops on a LEAKAGE ALARM, the row and
   the alarms are printed above it: stop and report them before any other run (R16).

Commit this notebook with its outputs cleared (Edit > Clear All Outputs); it prints counts only,
never images or metadata records.

Data: Caltech Camera Traps (CCT20), licence
[CDLA-Permissive 1.0](https://cdla.dev/permissive-1-0/). Please cite: Sara Beery, Grant Van
Horn, Pietro Perona. "Recognition in Terra Incognita." Proceedings of the European Conference
on Computer Vision (ECCV), 2018. https://arxiv.org/abs/1807.04975. Code: MIT.

In [ ]:
# Parameters: set these, then Save & Run All.
from pathlib import Path

CONFIG_NAME = "small_cnn"  # small_cnn, resnet18_arm_a or resnet18_arm_b (config.yaml)
RUN_SEED = None  # None: the config's run seed (0); 1 or 2 for the extra seeds (plan.md section 6)
COMMIT = "dc05dad68ff8a5c99980e6dd6778375bb1f5557b"  # full sha of the main commit to run
DATA_ROOT = Path("/kaggle/input/cct20-512")  # holds annotations/, images/ and day_night.json

REPO_URL = "https://github.com/codx999/who-goes-there"  # the public repo
WORK = Path("/kaggle/working")  # kept as the version's output
CLONE = WORK / "who-goes-there"
OUT_DIR = WORK / "models"  # the checkpoint

## 1. Check the parameters and the GPU

Fails in seconds, before anything is cloned or installed.

In [ ]:
import re
import sys

import torch

print(f"Python {sys.version.split()[0]}, torch {torch.__version__}")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError(f"COMMIT must be a full 40-character commit sha, got {COMMIT!r}")
if CONFIG_NAME not in ("small_cnn", "resnet18_arm_a", "resnet18_arm_b"):
    raise ValueError(f"CONFIG_NAME must be a training config of config.yaml, got {CONFIG_NAME!r}")
if not torch.cuda.is_available():
    raise RuntimeError("no GPU: set Session options > Accelerator to a GPU, then run again")
print(f"GPU {torch.cuda.get_device_name(0)}")

## 2. Clone the repo at `COMMIT` and install `trailcam` from the clone

`config.yaml` and `docs/ledger.md` are not in the wheel, so `train.py` is given both from this
clone (`--config`, `--ledger`): code, config and config hash come from one commit. This is the
project's one `pip install` (the notebook exception in `CLAUDE.md`). The package's dependency
floors lie below Kaggle's versions, so pip keeps the preinstalled torch, torchvision, Pillow,
timm and PyYAML; the cell checks that it did.

In [ ]:
import subprocess

if CLONE.exists():  # a second run in one interactive session reuses the clone
    print(f"reusing the clone at {CLONE}")
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(CLONE)], check=True)
    subprocess.run(["git", "-C", str(CLONE), "checkout", "--quiet", "--detach", COMMIT], check=True)
head = subprocess.run(
    ["git", "-C", str(CLONE), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
changed = subprocess.run(
    ["git", "-C", str(CLONE), "status", "--porcelain", "--untracked-files=no"],
    check=True,
    capture_output=True,
    text=True,
).stdout.split("\n")
changed = [line for line in changed if line and not line.endswith(" docs/ledger.md")]
if head != COMMIT or changed:
    raise RuntimeError(
        f"{CLONE} is at {head} with {len(changed)} changed files, not a clean {COMMIT}: "
        "delete it (or restart the session) and run again"
    )
print(f"commit {head}")

In [ ]:
from importlib.metadata import version

KEPT = ("torch", "torchvision", "Pillow", "timm", "PyYAML")  # Kaggle's preinstalled builds
preinstalled = {name: version(name) for name in KEPT}
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(CLONE)], check=True)
after_install = {name: version(name) for name in KEPT}
if after_install != preinstalled:
    raise RuntimeError(f"pip replaced preinstalled packages: {preinstalled} -> {after_install}")
print(f"trailcam {version('trailcam')} from commit {head}; kept {after_install}")

## 3. Check the installed code and the cache

Every module of the installed `trailcam` must match the clone byte for byte. The cache must hold
the annotations, the images and the day/night sidecar; it is loaded once here and shown as
counts only (images per split, images dropped per rule).

In [ ]:
import trailcam
from trailcam.data.metadata import DAY_NIGHT_SIDECAR, load_metadata
from trailcam.train import train
from trailcam.train.train import CACHE_ANNOTATIONS, CACHE_IMAGES, file_sha256

installed = Path(trailcam.__file__).parent
modules = sorted(p.relative_to(CLONE / "trailcam") for p in (CLONE / "trailcam").rglob("*.py"))
differ = [
    str(m) for m in modules if file_sha256(installed / m) != file_sha256(CLONE / "trailcam" / m)
]
if differ:
    raise RuntimeError(f"the installed trailcam differs from the clone in {differ}")
print(f"{len(modules)} trailcam modules at {installed} match commit {head}")

needed = {
    "annotations": DATA_ROOT / CACHE_ANNOTATIONS,
    "images": DATA_ROOT / CACHE_IMAGES,
    "day/night sidecar": DATA_ROOT / DAY_NIGHT_SIDECAR,
}
missing = [name for name, path in needed.items() if not path.exists()]
if missing:
    roots = sorted(
        str(p.parent)
        for depth in ("*", "*/*", "*/*/*")
        for p in Path("/kaggle/input").glob(f"{depth}/{DAY_NIGHT_SIDECAR}")
    )
    raise FileNotFoundError(
        f"DATA_ROOT {DATA_ROOT} lacks the {', '.join(missing)}; folders under /kaggle/input "
        f"holding a {DAY_NIGHT_SIDECAR}: {roots or 'none (is the dataset attached?)'}"
    )
meta = load_metadata(needed["annotations"], needed["day/night sidecar"])
print(meta)
print(f"{sum(p.is_file() for p in needed['images'].rglob('*'))} files under {CACHE_IMAGES}/")

## 4. Train

`trailcam.train.train.main`, the same entry point as `python -m trailcam.train.train`. It checks
the out dir and the ledger before training, seeds `random`, NumPy and torch, trains with early
stopping on val macro-F1, restores and saves the best epoch, scores trans_val once (reported,
never selected on), prints the row and any leakage alarm, then appends the row to the clone's
ledger. It never reads cis_test or trans_test. A leakage alarm makes this cell fail after the
row is printed (R16).

In [ ]:
LEDGER = CLONE / "docs" / "ledger.md"
ledger_before = LEDGER.read_text(encoding="utf-8").splitlines()
argv = ["--config-name", CONFIG_NAME, "--config", str(CLONE / "config.yaml")]
argv += ["--data-root", str(DATA_ROOT), "--out-dir", str(OUT_DIR), "--ledger", str(LEDGER)]
if RUN_SEED is not None:
    argv += ["--run-seed", str(RUN_SEED)]
print("python -m trailcam.train.train " + " ".join(argv))
try:
    exit_code = train.main(argv)
except SystemExit as stop:  # argparse refused an argument or a path, before training
    raise RuntimeError(f"train.py stopped before training (exit {stop.code}), see above") from None
if exit_code == train.ALARM_EXIT:
    raise RuntimeError(
        "LEAKAGE ALARM (R16): the row and the alarms are printed above and the row is in the "
        "clone's ledger; stop and report them before any other run"
    )
if exit_code != 0:
    raise RuntimeError(f"train.py exited {exit_code}")

## 5. The ledger row

The one row the run appended to the clone's ledger, checked against `LEDGER_COLUMNS`, and the
checkpoint's SHA-256 computed again from the file kept under `OUT_DIR`. Paste the row into
`docs/ledger.md`.

In [ ]:
ledger_after = LEDGER.read_text(encoding="utf-8").splitlines()
new_rows = ledger_after[len(ledger_before) :]
if ledger_after[: len(ledger_before)] != ledger_before or len(new_rows) != 1:
    raise RuntimeError(f"expected one new row at the end of {LEDGER}, found {len(new_rows)}")
row = new_rows[0]
cells = [cell.strip() for cell in row.strip().strip("|").split("|")]
values = dict(zip(train.LEDGER_COLUMNS, cells, strict=True))
if train.ledger_row(values) != row:
    raise RuntimeError("the new ledger line does not read back as a row of LEDGER_COLUMNS")
checkpoint = OUT_DIR / f"{values['run id']}.pt"
digest = file_sha256(checkpoint)
if digest != values["checkpoint sha256"]:
    raise RuntimeError(f"{checkpoint} hashes to {digest}, the row says another hash")
print(f"commit {head}, config {CONFIG_NAME}, run seed {values['run seed']}")
print(f"checkpoint {checkpoint}, sha256 {digest}")
print("ledger row, for docs/ledger.md:")
print(train.ledger_header())
print(row)